In [10]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np
import re

# Load dataset dari out/
df = pd.read_csv('../out/cleandata_spotify.csv')
df = df.dropna(subset=['content_bersih'])
df = df[df['content_bersih'].str.split().str.len() >= 4]

# Ambil 1 dokumen per kelas sentimen
sample = pd.concat([
    df[df['sentiment_rating'] == s].head(1)
    for s in ['Negatif', 'Netral', 'Positif']
]).reset_index(drop=True)

corpus = sample['content_bersih'].tolist()
labels = sample['sentiment_rating'].tolist()

for i, (doc, label) in enumerate(zip(corpus, labels)):
    print(f'Doc {i} ({label}): {doc}')


Doc 0 (Negatif): cape nge download mslah premium
Doc 1 (Netral): overall bagus yah audio bagus fiturnya hari dengar notif premium gimana akan maksa download premium its bagus tolong ilangin fitur batas hari
Doc 2 (Positif): terima kasih sedia lagu lagu


In [11]:
# Representasi Vektor — fokus pada kata kunci ulasan Spotify
VOCAB = ['iklan', 'premium', 'lagu', 'fitur']

vectorizer = CountVectorizer(vocabulary=VOCAB)
X = vectorizer.fit_transform(corpus)

df_vec = pd.DataFrame(
    X.toarray(),
    columns=VOCAB,
    index=[f'Doc {i} ({labels[i]})' for i in range(len(corpus))]
)

print('=== MATRIKS VEKTOR (CountVectorizer) ===')
print(f'Vocabulary: {VOCAB}\n')
display(df_vec)


=== MATRIKS VEKTOR (CountVectorizer) ===
Vocabulary: ['iklan', 'premium', 'lagu', 'fitur']



,iklan,premium,lagu,fitur
Doc 0 (Negatif),0,1,0,0
Doc 1 (Netral),0,2,0,1
Doc 2 (Positif),0,0,2,0


In [12]:
# Cek kata-kata unik per dokumen
for i, doc in enumerate(corpus):
    words = set(doc.lower().split())
    print(f'Doc {i} ({labels[i]}): {sorted(words)}')


Doc 0 (Negatif): ['cape', 'download', 'mslah', 'nge', 'premium']
Doc 1 (Netral): ['akan', 'audio', 'bagus', 'batas', 'dengar', 'download', 'fitur', 'fiturnya', 'gimana', 'hari', 'ilangin', 'its', 'maksa', 'notif', 'overall', 'premium', 'tolong', 'yah']
Doc 2 (Positif): ['kasih', 'lagu', 'sedia', 'terima']


In [13]:
# Hitung Cosine Similarity terhadap Doc 0
cos_sim = cosine_similarity(X[0:1], X)

df_cs = pd.DataFrame(X.toarray(), columns=VOCAB)
df_cs.index = [f'Doc {i} ({labels[i]})' for i in range(len(corpus))]
df_cs['Cosine Sim to Doc 0'] = cos_sim.flatten()

print('=== COSINE SIMILARITY (CountVectorizer) ===')
display(df_cs)


=== COSINE SIMILARITY (CountVectorizer) ===


,iklan,premium,lagu,fitur,Cosine Sim to Doc 0
Doc 0 (Negatif),0,1,0,0,1.000000
Doc 1 (Netral),0,2,0,1,0.894427
Doc 2 (Positif),0,0,2,0,0.000000


In [14]:
# Preprocessing: corpus sudah bersih dari kolom content_bersih
# Tampilkan kembali dokumen yang digunakan
for i, doc in enumerate(corpus):
    print(f'Doc {i} ({labels[i]}): {doc}')


Doc 0 (Negatif): cape nge download mslah premium
Doc 1 (Netral): overall bagus yah audio bagus fiturnya hari dengar notif premium gimana akan maksa download premium its bagus tolong ilangin fitur batas hari
Doc 2 (Positif): terima kasih sedia lagu lagu


**CountVectorizer:** Mengubah teks menjadi angka (matriks frekuensi kata). Di sini kita membatasi fitur pada kata kunci ulasan Spotify (`iklan`, `premium`, `lagu`, `fitur`) agar dimensi vektor tetap kecil dan mudah divisualisasikan.

**Cosine Similarity vs Distance:** Menghasilkan nilai 0–1. Semakin mendekati 1, semakin mirip arah topik kedua dokumen, terlepas dari panjangnya.

**Sumber data:** `../out/cleandata_spotify.csv` — kolom `content_bersih` (hasil preprocessing minggu-3), 1 dokumen per kelas sentimen (Positif, Negatif, Netral).

In [15]:
from collections import Counter

# TF Matrix — semua kata dari corpus
tf = pd.DataFrame(
    [Counter(doc.split()) for doc in corpus]
).fillna(0).astype(int)
tf.index = [f'Doc {i} ({labels[i]})' for i in range(len(corpus))]

print('=== MATRIKS TERM FREQUENCY (TF) ===')
display(tf)


=== MATRIKS TERM FREQUENCY (TF) ===


,cape,nge,download,mslah,premium,overall,bagus,yah,audio,fiturnya,...,maksa,its,tolong,ilangin,fitur,batas,terima,kasih,sedia,lagu
Doc 0 (Negatif),1,1,1,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Doc 1 (Netral),0,0,1,0,2,1,3,1,1,1,...,1,1,1,1,1,1,0,0,0,0
Doc 2 (Positif),0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,1,1,2


In [17]:
def length(v):
    return np.sqrt((v ** 2).sum())

def cos_dist(v, w):
    return 1 - (v * w).sum() / (length(v) * length(w))

# Cosine distance antar dokumen berdasarkan TF penuh
n = len(corpus)
print('=== COSINE DISTANCE (TF full vocab) ===')
for i in range(n):
    for j in range(i+1, n):
        d = cos_dist(tf.iloc[i], tf.iloc[j])
        print(f'Doc {i} ({labels[i]}) ↔ Doc {j} ({labels[j]}): {d:.4f}')


=== COSINE DISTANCE (TF full vocab) ===
Doc 0 (Negatif) ↔ Doc 1 (Netral): 0.7628
Doc 0 (Negatif) ↔ Doc 2 (Positif): 1.0000
Doc 1 (Netral) ↔ Doc 2 (Positif): 1.0000


In [18]:
def length(v):
    return np.sqrt((v ** 2).sum())

def cos_dist(v, w):
    denom = length(v) * length(w)
    return float('nan') if denom == 0 else 1 - (v * w).sum() / denom

# Cosine distance menggunakan TF full vocab
n = len(corpus)
print('=== COSINE DISTANCE (TF full vocab) ===')
for i in range(n):
    for j in range(i + 1, n):
        d = cos_dist(tf.iloc[i], tf.iloc[j])
        print(f'Doc {i} ({labels[i]}) ↔ Doc {j} ({labels[j]}): {d:.4f}')


=== COSINE DISTANCE (TF full vocab) ===
Doc 0 (Negatif) ↔ Doc 1 (Netral): 0.7628
Doc 0 (Negatif) ↔ Doc 2 (Positif): 1.0000
Doc 1 (Netral) ↔ Doc 2 (Positif): 1.0000


In [19]:
# TF Matrix hanya untuk kata kunci VOCAB (versi ringkas)
tf_vocab = pd.DataFrame(
    [Counter(doc.split()) for doc in corpus],
    columns=VOCAB
).fillna(0).astype(int)
tf_vocab.index = [f'Doc {i} ({labels[i]})' for i in range(len(corpus))]

print(f'Matriks Term Frequency (vocab: {VOCAB}):')
display(tf_vocab)


Matriks Term Frequency (vocab: ['iklan', 'premium', 'lagu', 'fitur']):


,iklan,premium,lagu,fitur
Doc 0 (Negatif),0,1,0,0
Doc 1 (Netral),0,2,0,1
Doc 2 (Positif),0,0,2,0


In [20]:
full_cos_sim_matrix = cosine_similarity(X)

index_labels = [f'Doc {i} ({labels[i]})' for i in range(len(corpus))]
cos_sim_df = pd.DataFrame(full_cos_sim_matrix, index=index_labels, columns=index_labels)

print('=== FULL COSINE SIMILARITY MATRIX ===')
display(cos_sim_df)


=== FULL COSINE SIMILARITY MATRIX ===


,Doc 0 (Negatif),Doc 1 (Netral),Doc 2 (Positif)
Doc 0 (Negatif),1.000000,0.894427,0.0
Doc 1 (Netral),0.894427,1.000000,0.0
Doc 2 (Positif),0.000000,0.000000,1.0


**Summary**

Data diambil dari `../out/cleandata_spotify.csv` — 1 sampel per kelas sentimen (Positif, Negatif, Netral) menggunakan kolom `content_bersih`.

Vocabulary difokuskan pada kata kunci ulasan Spotify (`iklan`, `premium`, `lagu`, `fitur`) untuk menjaga representasi vektor tetap interpretatif. Cosine similarity antar dokumen mencerminkan kesamaan topik: ulasan yang sama-sama menyebut kata-kata tersebut akan berada lebih dekat dalam ruang vektor, tanpa terpengaruh panjang ulasan.